# Random Forest Classifier (Scratch, Tanpa scikit-learn) — Versi Rapi per Sel

Notebook ini implementasi **Random Forest Classifier** dari nol (tanpa `sklearn`), lengkap dari awal:
1. **Read CSV** pakai `pd.read_csv()`
2. Set **fitur (X)** dan **target (Y)**
3. Fungsi bantu (split, gini, bootstrap, akurasi)
4. **Decision Tree (CART + Gini)** sebagai base learner (dipecah per sel)
5. **Random Forest** (bagging + feature subsampling)
6. Training, evaluasi, prediksi data baru

> **PENTING:** pastikan file CSV kamu ada di folder yang sama dengan notebook ini.
- Kalau kamu pakai dataset yang aku buat: nama file-nya **`data.csv`**


## 1) Import library

In [ ]:
import pandas as pd
import numpy as np

## 2) Baca data (pd.read_csv)

In [ ]:
# Ganti nama file CSV kamu di sini
df = pd.read_csv("data.csv")

df.head()

## 3) Pilih fitur (X) dan target (Y)

In [ ]:
# GANTI sesuai kolom dataset kamu
FEATURES = ["luas_tanah_m2", "luas_bangunan_m2"]
TARGET = "kelas"

X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy()

print("Jumlah data:", len(df))
print("Fitur:", FEATURES)
print("Target:", TARGET)
print("Label unik:", np.unique(y))

## 4) Fungsi bantu: split data, gini, majority, akurasi

In [ ]:
def train_test_split(X, y, test_size=0.2, seed=42):
    """Split data acak: train & test."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(X))
    rng.shuffle(idx)
    split = int(len(X) * (1 - test_size))
    train_idx, test_idx = idx[:split], idx[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def gini(y):
    """Gini impurity."""
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)

def majority_class(y):
    """Kelas mayoritas."""
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## 5) Fungsi bantu Random Forest: bootstrap sampling & majority vote

In [ ]:
def bootstrap_sample(X, y, rng):
    """Ambil sampel dengan pengembalian (bagging)."""
    n = len(X)
    idx = rng.integers(0, n, size=n)
    return X[idx], y[idx]

def majority_vote(pred_matrix):
    """pred_matrix shape: (n_trees, n_samples) -> output (n_samples,)"""
    n_trees, n_samples = pred_matrix.shape
    out = []
    for j in range(n_samples):
        values, counts = np.unique(pred_matrix[:, j], return_counts=True)
        out.append(values[np.argmax(counts)])
    return np.array(out)

## 6) Struktur Node (wadah simpul tree)

In [ ]:
class Node:
    def __init__(self, *, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # jika leaf, berisi label

## 7) Decision Tree (base learner) — kerangka class

In [ ]:
class DecisionTreeClassifierScratch:
    """Decision Tree sederhana (CART + Gini) untuk dipakai di Random Forest."""

    def __init__(self, max_depth=5, min_samples_split=2, min_impurity_decrease=1e-7, max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease
        self.max_features = max_features  # jumlah fitur yang dicoba tiap split (feature subsampling)
        self.rng = np.random.default_rng(random_state)
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

    def _predict_one(self, x, node):
        while node.value is None:
            if x[node.feature_idx] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

## 8) Decision Tree — cari split terbaik (dengan feature subsampling)

In [ ]:
def best_split_tree(X, y, rng, max_features=None):
    n_samples, n_features = X.shape
    parent_impurity = gini(y)

    best_gain = 0.0
    best_feature = None
    best_thresh = None

    # pilih subset fitur untuk dicoba (Random Forest style)
    feature_indices = np.arange(n_features)
    if max_features is not None:
        k = max(1, min(int(max_features), n_features))
        feature_indices = rng.choice(feature_indices, size=k, replace=False)

    for f in feature_indices:
        values = np.unique(X[:, f])
        if len(values) == 1:
            continue
        thresholds = (values[:-1] + values[1:]) / 2.0

        for t in thresholds:
            left_mask = X[:, f] <= t
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            y_left, y_right = y[left_mask], y[right_mask]
            w_left = len(y_left) / n_samples
            w_right = len(y_right) / n_samples
            child_impurity = w_left * gini(y_left) + w_right * gini(y_right)

            gain = parent_impurity - child_impurity
            if gain > best_gain:
                best_gain = gain
                best_feature = f
                best_thresh = t

    return best_feature, best_thresh, best_gain

## 9) Decision Tree — build tree rekursif

In [ ]:
def _best_split(self, X, y):
    return best_split_tree(X, y, rng=self.rng, max_features=self.max_features)

DecisionTreeClassifierScratch._best_split = _best_split

def _build_tree(self, X, y, depth):
    # stop: semua label sama
    if len(np.unique(y)) == 1:
        return Node(value=y[0])

    # stop: depth mentok / data sedikit
    if depth >= self.max_depth or len(y) < self.min_samples_split:
        return Node(value=majority_class(y))

    feature, thresh, gain = self._best_split(X, y)

    # stop: split tidak bagus
    if feature is None or gain < self.min_impurity_decrease:
        return Node(value=majority_class(y))

    left_mask = X[:, feature] <= thresh
    right_mask = ~left_mask

    left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
    right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

    return Node(feature_idx=feature, threshold=thresh, left=left, right=right)

DecisionTreeClassifierScratch._build_tree = _build_tree

## 10) Random Forest Classifier (bagging + feature subsampling)

In [ ]:
class RandomForestClassifierScratch:
    def __init__(self, n_estimators=25, max_depth=5, min_samples_split=2,
                 max_features="sqrt", random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.rng = np.random.default_rng(random_state)
        self.trees = []

    def _resolve_max_features(self, n_features):
        if self.max_features is None:
            return None
        if isinstance(self.max_features, int):
            return self.max_features
        if isinstance(self.max_features, float):
            return max(1, int(round(self.max_features * n_features)))
        if isinstance(self.max_features, str):
            mf = self.max_features.lower()
            if mf == "sqrt":
                return max(1, int(np.sqrt(n_features)))
            if mf == "log2":
                return max(1, int(np.log2(n_features)))
            if mf == "all":
                return n_features
        # fallback
        return None

    def fit(self, X, y):
        self.trees = []
        n_features = X.shape[1]
        max_feat = self._resolve_max_features(n_features)

        for _ in range(self.n_estimators):
            Xb, yb = bootstrap_sample(X, y, self.rng)

            tree = DecisionTreeClassifierScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_feat,
                random_state=int(self.rng.integers(0, 1_000_000_000))
            )
            tree.fit(Xb, yb)
            self.trees.append(tree)

    def predict(self, X):
        # kumpulkan prediksi dari semua tree
        preds = np.array([t.predict(X) for t in self.trees])  # (n_trees, n_samples)
        return majority_vote(preds)

## 11) Training & evaluasi Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, seed=123)

rf = RandomForestClassifierScratch(
    n_estimators=35,
    max_depth=5,
    min_samples_split=3,
    max_features="sqrt",
    random_state=123
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy(y_test, y_pred))
pd.DataFrame({"y_true": y_test[:10], "y_pred": y_pred[:10]})

## Evaluasi tambahan: Confusion Matrix, Precision, Recall, F1 (tanpa sklearn)

In [ ]:
def confusion_matrix_df(y_true, y_pred, labels=None):
    """Confusion matrix dalam bentuk DataFrame."""
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    labels = list(labels)
    label_to_idx = {lab: i for i, lab in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        cm[label_to_idx[yt], label_to_idx[yp]] += 1
    return pd.DataFrame(cm, index=[f"True_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])

def classification_report_df(y_true, y_pred, labels=None):
    """Hitung precision/recall/F1 per kelas + macro & weighted avg."""
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    labels = list(labels)
    
    rows = []
    support_total = 0
    macro_p = macro_r = macro_f1 = 0.0
    weighted_p = weighted_r = weighted_f1 = 0.0
    
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        support = np.sum(y_true == lab)
        support_total += support
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        
        rows.append([lab, precision, recall, f1, support])
        
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        
        weighted_p += precision * support
        weighted_r += recall * support
        weighted_f1 += f1 * support
    
    n_classes = len(labels)
    macro_p /= n_classes
    macro_r /= n_classes
    macro_f1 /= n_classes
    
    if support_total > 0:
        weighted_p /= support_total
        weighted_r /= support_total
        weighted_f1 /= support_total
    
    report = pd.DataFrame(rows, columns=["class", "precision", "recall", "f1_score", "support"])
    
    # Tambahkan ringkasan
    summary = pd.DataFrame([
        ["macro_avg", macro_p, macro_r, macro_f1, support_total],
        ["weighted_avg", weighted_p, weighted_r, weighted_f1, support_total],
    ], columns=report.columns)
    
    report = pd.concat([report, summary], ignore_index=True)
    return report

# --- Hitung metrik ---
labels = np.unique(np.concatenate([y_test, y_pred]))
cm = confusion_matrix_df(y_test, y_pred, labels=labels)
report = classification_report_df(y_test, y_pred, labels=labels)

print("Akurasi:", accuracy(y_test, y_pred))
print("\nConfusion Matrix:")
display(cm)

print("\nPrecision / Recall / F1 per kelas:")
display(report)

## 12) Prediksi data baru (contoh)

In [ ]:
contoh = np.array([
    [80, 60],
    [150, 110],
], dtype=float)

pred_contoh = rf.predict(contoh)
pd.DataFrame(contoh, columns=FEATURES).assign(prediksi_kelas=pred_contoh)